# NB00 · City digital twin + simulated episode dataset  (CPU · ~10–25 min)

**Role in the pipeline:** produces everything the other notebooks consume — `w72_common.py` (shared code), the episodes (`data/<split>/*.npz`), and `w72_manifest.csv`.
**Run this FIRST.** Then attach this notebook's *output* as an input to NB01–NB05 (Add Input → Notebook Output Files, or *Save Version → Create Dataset from output*).

What it does
1. Writes `w72_common.py` (simulator, fault injector, features, Hawkes/ETAS maths).
2. Simulates `train/val/test` (anchor city), `heldout` (unseen city with different topography/drains/grid/climate) and `drift` (paired drain-capacity-collapse vs control) episodes — 10 days × hourly × 20/16 zones × 10 layers.
3. Hidden ground-truth coupling `G_true` is stored for **scoring only**.
4. Sanity checks + diagnostic plots (label base rates, cascade ordering, forecast-error growth, fault profiles, city maps).

> Simulator mechanics are queues / SCS-style runoff / thresholds / SEIR / Gaussian puff / ETAS — *not* Hawkes. The models never see the simulator equations.
> Checkpointing: finished episodes are skipped on re-run, so you can resume after a session drop.

In [ ]:
%%writefile w72_common.py
"""
w72_common.py  --  shared code for THE 72-HOUR WARNING (Cascade'26, Theme 2 / Problem 2)

Sections
  1. constants / config
  2. city digital twin
  3. simulator (mechanistic, deliberately NOT Hawkes-shaped)
  4. labels  (TRAINING / EVALUATION ONLY - never import from inference code)
  5. observation model + fault injector (F0..F5), as-of semantics
  6. event extraction + discrete-time Hawkes (numpy, analytic gradients)
  7. ETAS
  8. feature builder for the risk head
"""
import numpy as np, json, math, os
from numpy.lib.stride_tricks import sliding_window_view
from scipy.special import ndtr

# ----------------------------------------------------------------------------- 1. constants
LAYERS = ['WX', 'HYD', 'TRF', 'PWR', 'EMS', 'HOS', 'HLT', 'AIR', 'SEI', 'COM']
LI = {n: i for i, n in enumerate(LAYERS)}
VARS = ['rain_mm_h', 'wbgt', 'drain_load', 'water_depth_cm', 'speed_idx', 'load_ratio', 'pct_out',
        'ems_resp_min', 'bed_occ', 'ed_ratio', 'conc_idx', 'damage_idx', 'shaking_mmi', 'com_load',
        'report_rate', 'temp_c']
VI = {n: i for i, n in enumerate(VARS)}
NV = len(VARS)
VAR_DEFAULT = np.array([0, 24, .2, 0, .9, .7, 0, 10, .7, 1, 0, 0, 1, .2, .5, 28], np.float32)
VAR_MIN = np.array([0, 10, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0, 0, -5], np.float32)
VAR_MAX = np.array([150, 45, 10, 200, 1, 2, 1, 120, 1.3, 12, 50, 1, 12, 1.5, 40, 55], np.float32)
NOISE_SIGMA = np.array([.3, .4, .03, .8, .02, .02, .01, .8, .01, .08, .05, .01, .15, .02, .3, .3], np.float32)
BASE_DELAY = np.array([0, 0, 0, 0, 0, 0, 1, 0, 1, 2, 0, 3, 0, 0, 0, 0], np.int32)   # hours
TH = (0.35, 0.60, 0.85)                       # severity ladder thresholds on stress: 0 normal,1 stressed,2 degraded,3 failed
EVENT_TYPES = [f'{l}_{s}' for l in LAYERS for s in ('deg', 'fail')]   # 20 point-process types
NTYPE = len(EVENT_TYPES)
CRIT_NAMES = ['access', 'power', 'hospital', 'heat', 'contam', 'structural', 'comms']
CRIT_K = np.array([2, 2, 2, 3, 2, 2, 2])      # sustained hours required
HORIZONS = [1, 3, 6, 12, 24, 48, 72]
HAZARDS = ['flood', 'heat', 'grid', 'epidemic', 'quake', 'chem', 'compoundA', 'compoundB', 'none']
COMPONENTS = {'flood': {'flood'}, 'heat': {'heat'}, 'grid': {'grid'}, 'epidemic': {'epi'}, 'quake': {'quake'},
              'chem': {'chem'}, 'compoundA': {'flood', 'grid', 'epi'}, 'compoundB': {'quake', 'chem', 'grid'}, 'none': set()}
HAZARD_W = np.array([.18, .15, .13, .12, .10, .08, .09, .06, .09]); HAZARD_W = HAZARD_W / HAZARD_W.sum()
PROFILE_NAMES = ['F0', 'F1', 'F2', 'F3', 'F4', 'F5']
TAUS = np.array([2., 8., 24., 72.])           # Hawkes lag basis (hours); 72 h added to cover multi-day epidemic lags

# hidden ground-truth coupling (layer-level, from -> to, lag range hours, mechanism).  Used ONLY for scoring.
GEDGES = [
    ('WX', 'HYD', 0, 3, 'physical'), ('WX', 'PWR', 0, 6, 'physical'), ('WX', 'EMS', 1, 6, 'logical'),
    ('WX', 'HOS', 12, 48, 'physical'), ('WX', 'HLT', 12, 48, 'physical'),
    ('HYD', 'TRF', 0, 4, 'physical'), ('HYD', 'PWR', 2, 8, 'physical'), ('HYD', 'EMS', 0, 4, 'logical'),
    ('HYD', 'COM', 0, 3, 'logical'), ('TRF', 'EMS', 0, 4, 'physical'),
    ('PWR', 'TRF', 0, 2, 'physical'), ('PWR', 'HYD', 1, 4, 'physical'), ('PWR', 'COM', 4, 8, 'cyber'),
    ('PWR', 'HOS', 6, 24, 'physical'), ('EMS', 'HOS', 1, 6, 'logical'), ('HOS', 'EMS', 2, 8, 'logical'),
    ('EMS', 'COM', 0, 2, 'cyber'), ('HLT', 'HOS', 24, 96, 'logical'), ('SEI', 'PWR', 0, 3, 'physical'),
    ('SEI', 'TRF', 0, 2, 'physical'), ('SEI', 'AIR', 1, 6, 'physical'), ('SEI', 'HOS', 1, 6, 'logical'),
    ('SEI', 'EMS', 0, 3, 'logical'), ('SEI', 'COM', 0, 2, 'cyber'), ('AIR', 'TRF', 1, 3, 'physical'),
    ('AIR', 'HOS', 1, 6, 'logical'), ('AIR', 'EMS', 1, 4, 'logical'), ('AIR', 'COM', 0, 2, 'cyber'),
]
G_EDGE_ARR = np.array([[LI[a], LI[b], lo, hi] for a, b, lo, hi, _ in GEDGES])
def g_true_matrix():
    G = np.zeros((10, 10), bool)                 # G[from, to]
    for a, b, *_ in GEDGES: G[LI[a], LI[b]] = True
    return G
def eval_pair_mask():
    """layer pairs (from,to) the model is allowed to be scored on: off-diagonal, target not WX/SEI (exogenous)"""
    M = ~np.eye(10, dtype=bool); M[:, LI['WX']] = False; M[:, LI['SEI']] = False
    return M
def forbidden_type_mask():
    """A_mask[i(target type), j(source type)]: 0 = physically impossible (-> WX, -> SEI). Masks FORBID, never assert."""
    m = np.ones((NTYPE, NTYPE), np.float32)
    for l in ('WX', 'SEI'): m[2 * LI[l]:2 * LI[l] + 2, :] = 0
    return m

# ----------------------------------------------------------------------------- 2. city twin
def make_city(kind='anchor', seed=None):
    seed = {'anchor': 11, 'heldout': 97}[kind] if seed is None else seed
    rng = np.random.default_rng(seed)
    nx, ny = (5, 4) if kind == 'anchor' else (4, 4)
    ix, iy = np.meshgrid(np.arange(nx), np.arange(ny)); ix, iy = ix.ravel(), iy.ravel()
    Z = nx * ny; coords = np.c_[ix, iy].astype(float) * 3.0
    adj = np.zeros((Z, Z), np.float32)
    for a in range(Z):
        for b in range(Z):
            if abs(ix[a] - ix[b]) + abs(iy[a] - iy[b]) == 1: adj[a, b] = 1
    W = adj / adj.sum(1, keepdims=True)
    if kind == 'anchor': elev = 40 * ix / max(ix.max(), 1) + 10 * iy / max(iy.max(), 1)          # tilted plain
    else: elev = 35 * np.sqrt(((ix - ix.mean()) ** 2 + (iy - iy.mean()) ** 2)) / 2.0              # bowl -> water collects centrally
    elev = elev + rng.normal(0, 5, Z); elev = 0.5 * elev + 0.5 * (W @ elev)
    down = -np.ones(Z, int)
    for z in range(Z):
        nb = np.where(adj[z] > 0)[0]; lo = nb[np.argmin(elev[nb])]
        if elev[lo] < elev[z]: down[z] = lo
    drain_cap = (7.0 if kind == 'anchor' else 5.0) * rng.lognormal(0, 0.25, Z)
    sub = (ix // 2) + ((nx + 1) // 2) * (iy // 2) if kind == 'anchor' else ix.copy()               # different feeder topology
    nsub = sub.max() + 1
    pop = rng.lognormal(0, 0.5, Z); pop = pop / pop.mean()
    nh = 4 if kind == 'anchor' else 3
    hosp = np.sort(np.argsort(-pop)[:nh] if kind == 'anchor' else rng.choice(Z, nh, replace=False))
    beds = rng.uniform(220, 420, nh) if kind == 'anchor' else rng.uniform(150, 300, nh)
    gen_hours = rng.uniform(8, 24, nh)
    dist_h = np.linalg.norm(coords[:, None] - coords[hosp][None], axis=2)
    hos_of = dist_h.argmin(1)
    nn_h = np.array([np.argsort(np.linalg.norm(coords[hosp] - coords[hosp[h]], axis=1))[1] for h in range(nh)])
    rating = rng.uniform(0.92, 1.18, nsub) if kind == 'anchor' else rng.uniform(0.82, 1.02, nsub)
    popsub = np.array([pop[sub == s].mean() for s in range(nsub)])
    sub_demand = 0.9 + 0.2 * popsub / popsub.mean()
    pct = lambda a: (np.argsort(np.argsort(a)) / max(len(a) - 1, 1)).astype(np.float32)
    catch_pop = np.array([pop[hos_of == h].sum() for h in range(nh)])
    bed_density = (beds / catch_pop)[hos_of]
    n_up = np.array([(down == z).sum() for z in range(Z)])
    static = np.stack([pct(elev), pct(pop), pct(bed_density), pct(drain_cap), np.isin(np.arange(Z), hosp).astype(np.float32),
                       n_up / max(n_up.max(), 1), pct(dist_h.min(1))], 1).astype(np.float32)
    UP = np.zeros((Z, Z), np.float32)
    for y in range(Z):
        if down[y] >= 0: UP[down[y], y] = 1
    UP = UP / np.maximum(UP.sum(1, keepdims=True), 1)
    SUB = (sub[:, None] == sub[None, :]).astype(np.float32); SUB = SUB / SUB.sum(1, keepdims=True)
    return dict(kind=kind, Z=Z, coords=coords, adj=adj, W=W, elev=elev, down=down, drain_cap=drain_cap, sub=sub, nsub=nsub,
                pop=pop, hosp=hosp, beds=beds, gen_hours=gen_hours, hos_of=hos_of, nn_h=nn_h, rating=rating,
                sub_demand=sub_demand, static=static, UP=UP, SUB=SUB,
                Tbase=29.0 if kind == 'anchor' else 24.0, RH=65.0 if kind == 'anchor' else 75.0,
                rain_scale=1.0 if kind == 'anchor' else 0.85, heat_scale=1.0 if kind == 'anchor' else 1.25)
STATIC_NAMES = ['elev_pct', 'pop_pct', 'bed_density_pct', 'drain_cap_pct', 'is_hospital', 'n_upstream', 'dist_hospital_pct']

# ----------------------------------------------------------------------------- 3. simulator
def wbgt_stull(T, RH):
    """Stull (2011) wet-bulb approximation; WBGT(shade) ~ 0.7 Tw + 0.3 T  (documented simplification)"""
    Tw = (T * np.arctan(0.151977 * np.sqrt(RH + 8.313659)) + np.arctan(T + RH) - np.arctan(RH - 1.676331)
          + 0.00391838 * RH ** 1.5 * np.arctan(0.023101 * RH) - 4.686035)
    return 0.7 * Tw + 0.3 * T

ETAS_TRUE = dict(K=0.012, alpha=0.8, c=0.02, p=1.15, b=1.0, m0=4.0)    # K: per day^(p-1) units, time in DAYS
def etas_simulate(t_main, m_main, theta, t_end, rng, mu=0.0, m_max=8.0, window=30.0):
    """Branching simulation (days). Returns array (n,2) [time, mag] sorted. mu>0 adds background events."""
    K, al, c, p, b, m0 = (theta[k] for k in ('K', 'alpha', 'c', 'p', 'b', 'm0'))
    def F(Tw): return (c ** (1 - p) - (Tw + c) ** (1 - p)) / (p - 1)
    def mags(n): return np.minimum(m0 + rng.exponential(1 / (b * np.log(10)), n), m_max)
    queue = []
    if t_main is not None: queue.append((t_main, m_main))
    if mu > 0:
        nb = rng.poisson(mu * t_end); queue += list(zip(rng.uniform(0, t_end, nb), mags(nb)))
    ev = []
    while queue:
        t, m = queue.pop(); ev.append((t, m))
        Tw = min(window, t_end - t)
        if Tw <= 0: continue
        n = rng.poisson(K * 10 ** (al * (m - m0)) * F(Tw))
        if n:
            u = rng.random(n); s = ((1 - u) * c ** (1 - p) + u * (Tw + c) ** (1 - p)) ** (1 / (1 - p)) - c
            queue += list(zip(t + s, mags(n)))
    ev = np.array(sorted(ev)) if ev else np.zeros((0, 2))
    return ev

def _forecasts(rain, Tair, rng, T):
    """Weather forecasts ISSUED every 6 h with lead-dependent error growth. Never exposes the true future to a model:
    fc[t] only uses the forecast issued at floor(t/6)*6."""
    Z = rain.shape[1]; L = 78; issues = np.arange(0, T, 6)
    rp = np.vstack([rain, np.zeros((L, Z))]); Tp = np.concatenate([Tair, np.full(L, Tair[-1])])
    fc = np.zeros((T, Z, 6), np.float32); fs_r = np.zeros((len(issues), L), np.float32); fs_t = np.zeros((len(issues), L), np.float32)
    lead = np.arange(L); sig = 0.20 + 0.012 * lead
    for i, ti in enumerate(issues):
        bias = np.exp(rng.normal(0, 0.08 + 0.004 * lead))
        fr = rp[ti:ti + L] * np.exp(rng.normal(0, 1, (L, Z)) * sig[:, None]) * bias[:, None]
        ft = Tp[ti:ti + L] + rng.normal(0, 1) * (0.4 + 0.02 * lead) + rng.normal(0, 0.3, L)
        fs_r[i] = fr.mean(1); fs_t[i] = ft
        csr = np.vstack([np.zeros((1, Z)), np.cumsum(fr, 0)]); cst = np.concatenate([[0], np.cumsum(np.maximum(0, ft - 30))])
        for t in range(ti, min(ti + 6, T)):
            o = t - ti
            for j, h in enumerate((6, 24, 72)):
                fc[t, :, j] = csr[o + h] - csr[o]; fc[t, :, 3 + j] = cst[o + h] - cst[o]
    return fc, fs_r, fs_t

def stress_from_vars(X):
    """layer stress in [0,1.5] from the 16 physical/observed variables (same function for truth and for observed streams)"""
    g = lambda n: X[..., VI[n]]
    S = np.empty(X.shape[:-1] + (10,), np.float32)
    S[..., 0] = np.maximum(g('rain_mm_h') / 25.0, (g('wbgt') - 26.0) / 8.0)
    S[..., 1] = g('water_depth_cm') / 25.0
    S[..., 2] = (1 - g('speed_idx')) / 0.8
    S[..., 3] = np.maximum(g('pct_out'), (g('load_ratio') - 0.75) / 0.4)
    S[..., 4] = (g('ems_resp_min') - 10) / 30.0
    S[..., 5] = (g("bed_occ") - 0.5) / 0.45
    S[..., 6] = (g('ed_ratio') - 1) / 3.0
    S[..., 7] = g('conc_idx') / 1.5
    S[..., 8] = np.maximum(1.3 * g('damage_idx'), (g('shaking_mmi') - 4) / 5.0)
    S[..., 9] = g('com_load')
    return np.clip(S, 0, 1.5)
def levels_from_stress(S): return np.digitize(S, TH).astype(np.int8)

def simulate(city, hazard, seed, T=240, drift_at=None):
    rng = np.random.default_rng(seed); Z = city['Z']; coords = city['coords']; W = city['W']; down = city['down']
    sub = city['sub']; nsub = city['nsub']; H = city['hosp']; nH = len(H); B = city['beds'].copy(); Cm = np.zeros((nH, Z), np.float32)
    Cm[city['hos_of'], np.arange(Z)] = 1
    comps = COMPONENTS[hazard]; ta = np.arange(T); hod = ta % 24; popr = city['pop']
    diur = 0.75 + 0.25 * np.sin(2 * np.pi * (hod - 14) / 24); rush = np.exp(-((hod - 8) / 2.0) ** 2) + np.exp(-((hod - 18) / 2.0) ** 2)
    meta = dict(hazard=hazard, city=city['kind'], seed=int(seed), drift_at=-1, trigger_t=-1)
    # ---- weather
    Tair = city['Tbase'] + 4 * np.sin(2 * np.pi * (hod - 9) / 24) + rng.normal(0, 0.4, T)
    RH = city['RH'] - 12 * np.sin(2 * np.pi * (hod - 9) / 24) + rng.normal(0, 2, T)
    if 'heat' in comps or 'grid' in comps:
        amp = rng.uniform(3, 8) * city['heat_scale'] if 'heat' in comps else rng.uniform(1, 6)
        d0 = int(rng.integers(66, 110)); dur = int(rng.integers(72, 140))
        ramp = np.clip((ta - d0) / 18, 0, 1) * np.clip((d0 + dur - ta) / 18, 0, 1)
        Tair += amp * ramp; RH += rng.uniform(0, 14) * ramp
        if 'heat' in comps: meta['trigger_t'] = d0
    RH = np.clip(RH, 25, 95); wbgt = wbgt_stull(Tair, RH)
    rain = np.zeros((T, Z)); storm = np.zeros(T)
    n_storm = 1 if ({'flood', 'grid'} & comps) else int(rng.random() < 0.2)
    for _ in range(n_storm):
        ts = int(rng.integers(62, 125)); D = int(rng.integers(30, 80)); e = min(T, ts + D)
        peak = rng.uniform(6, 30) * city['rain_scale'] if 'flood' in comps else rng.uniform(4, 12)
        ctr = rng.uniform(coords.min(0), coords.max(0)); rad = rng.uniform(6, 16)
        wz = 0.35 + 0.65 * np.exp(-((coords - ctr) ** 2).sum(1) / (2 * rad ** 2))
        shp = np.zeros(T); shp[ts:e] = np.sin(np.pi * (np.arange(ts, e) - ts) / D) ** 2
        rain += peak * shp[:, None] * wz[None] * rng.lognormal(0, 0.35, (T, Z)); storm += shp
        if meta['trigger_t'] < 0 and ('flood' in comps or 'grid' in comps): meta['trigger_t'] = ts
    rain += rng.exponential(0.05, (T, Z)) * (rng.random((T, Z)) < 0.05)
    fc, fs_r, fs_t = _forecasts(rain, Tair, rng, T)
    # ---- seismic (exogenous, precomputed)
    mmi_ts = np.ones((T, Z)); dam = np.zeros((T, Z))
    if 'quake' in comps:
        t0 = rng.uniform(60, 120); M = rng.uniform(5.8, 7.4); epi = rng.uniform(coords.min(0) - 3, coords.max(0) + 3)
        evs = etas_simulate(t0 / 24, M, ETAS_TRUE, T / 24, rng, m_max=M - 0.3)
        site = rng.normal(0, 0.3, Z); vuln = rng.uniform(0.6, 1.0, Z); d = np.linalg.norm(coords - epi, axis=1)
        for te, m in evs:
            ti = int(te * 24)
            if ti < T: mmi_ts[ti] = np.maximum(mmi_ts[ti], 1.5 * m + 2.6 - 3.3 * np.log10(d + 10) + site)
        mmi_ts = np.clip(mmi_ts, 1, 12); maxm = np.maximum.accumulate(mmi_ts, axis=0)
        dam = vuln * ndtr((np.log(maxm) - np.log(7.5)) / 0.18); meta.update(trigger_t=int(t0), quake_mag=float(M), n_aftershocks=int(len(evs)))
    # ---- air sources
    sources = []
    if 'chem' in comps:
        t0 = int(rng.integers(60, 120)); sources.append(dict(t0=t0, z=int(rng.integers(Z)), Q=rng.uniform(60, 180), dur=int(rng.integers(4, 9))))
        meta['trigger_t'] = t0 if meta['trigger_t'] < 0 else min(meta['trigger_t'], t0)
    u0 = rng.uniform(2, 7); th = rng.uniform(0, 2 * np.pi) + np.cumsum(rng.normal(0, 0.05, T)); kspread = rng.uniform(0.8, 2.2)
    Pf = np.zeros((0, 4)); leaked = np.zeros(Z, bool)
    # ---- epidemic
    gam, sig = 1 / 72, 1 / 36; I_bg = 2e-4; Ih = np.full((T, Z), I_bg)
    if 'epi' in comps:
        R0 = rng.uniform(2.6, 4.2); t_seed = int(rng.integers(10, 60)); seeds = rng.choice(Z, int(rng.integers(1, 4)), replace=False)
        Ep = np.zeros(Z); Ip = np.full(Z, I_bg); Sf = np.ones(Z) - Ip; meta['trigger_t'] = t_seed if meta['trigger_t'] < 0 else min(meta['trigger_t'], t_seed)
    # ---- state
    sat = rng.uniform(0.1, 0.35, Z) + (0.25 * rng.random() if 'flood' in comps else 0); Sst = np.zeros(Z); block = rng.uniform(0, 0.2, Z)
    spill_in = np.zeros(Z); cap0 = city['drain_cap'] * rng.lognormal(0, 0.1, Z)
    out_rem = np.zeros(nsub, int); out_streak = np.zeros(Z); zfrac = rng.uniform(0.85, 1.0, Z); rating = city['rating']
    forced = []
    if 'grid' in comps:
        g0 = int(rng.integers(70, 130)); meta['trigger_t'] = g0 if meta['trigger_t'] < 0 else min(meta['trigger_t'], g0)
        for s in rng.choice(nsub, int(rng.integers(1, min(4, nsub))), replace=False): forced.append((g0 + int(rng.integers(0, 24)), int(s)))
    base_calls = 1.0 + 2.0 * popr; units = base_calls.sum() * 0.9 / 0.5
    o0 = rng.uniform(0.62, 0.78, nH); walkin = 0.35 * B / 96.0
    ems_ss = 0.25 * (Cm @ base_calls) * 0.91; LOS = o0 * B / (walkin + ems_ss); Npat = o0 * B
    occ = o0.copy(); heat_exp = 0.0; speed = np.full(Z, 0.9); pct_out = np.zeros(Z); dam_prev = np.zeros(Z)
    X = np.zeros((T, Z, NV), np.float32)
    for t in range(T):
        if drift_at is not None and t == drift_at:
            cap0 = cap0 * 0.5; block = np.clip(block + 0.25, 0, 0.8); meta['drift_at'] = int(t)
        # hydrology (pumps depend on previous-hour outage)
        runoff = rain[t] * (0.15 + 0.75 * sat ** 2); sat = np.clip(sat * 0.995 + 0.012 * rain[t] - 0.004, 0, 1)
        block = np.clip(block + 0.0005 * rain[t] - 0.0002, 0, 0.8)
        cap = cap0 * (1 - block) * (1 - 0.6 * pct_out)
        inflow = runoff + spill_in; load_d = inflow / np.maximum(cap, 0.5)
        Sst = np.maximum(Sst + inflow - np.minimum(Sst + inflow, cap), 0)
        sp = np.where(Sst > 150, 0.4 * (Sst - 150), 0); Sst = Sst - sp
        spill_in = np.zeros(Z); m = down >= 0; np.add.at(spill_in, down[m], sp[m]); depth = 0.2 * Sst
        # air
        for s in sources:
            if s['t0'] <= t < s['t0'] + s['dur']: Pf = np.vstack([Pf, [coords[s['z'], 0], coords[s['z'], 1], s['Q'], 0]])
        conc = np.zeros(Z)
        if len(Pf):
            sg = 0.6 + kspread * Pf[:, 3]; d2 = ((coords[None] - Pf[:, None, :2]) ** 2).sum(-1)
            conc = (Pf[:, 2:3] / (2 * np.pi * sg[:, None] ** 2) * np.exp(-d2 / (2 * sg[:, None] ** 2))).sum(0)
            Pf[:, 0] += u0 * 3.6 * np.cos(th[t]); Pf[:, 1] += u0 * 3.6 * np.sin(th[t]); Pf[:, 3] += 1; Pf = Pf[Pf[:, 3] <= 24]
        if 'quake' in comps:
            for z in np.where((dam[t] > 0.5) & ~leaked)[0]:
                if rng.random() < 0.15: leaked[z] = True; sources.append(dict(t0=t + 1, z=int(z), Q=rng.uniform(20, 70), dur=5))
        # power
        n_out = int((out_rem > 0).sum()); out_rem = np.maximum(out_rem - 1, 0)
        load = 0.68 * diur[t] * (1 + 0.035 * max(0, Tair[t] - 27)) * city['sub_demand'] / rating * (1 + 0.25 * n_out / nsub)
        md = np.zeros(nsub); np.maximum.at(md, sub, depth); dsub = np.zeros(nsub); np.maximum.at(dsub, sub, dam[t])
        p = (0.0001 + 0.02 / (1 + np.exp(-(load - 1.0) / 0.04)) + 0.03 / (1 + np.exp(-(md - 20) / 3)) + 0.1 * dsub
             + (0.012 * storm[t] if 'grid' in comps else 0))
        fault = rng.random(nsub) < p
        for (tf, s) in forced:
            if tf == t: fault[s] = True
        fault &= (out_rem == 0)
        dur_f = np.minimum(np.ceil(rng.lognormal(np.log(4.5), 0.6, nsub) * (1 + 0.5 * n_out)), 48).astype(int)
        out_rem = np.where(fault, dur_f, out_rem); outf = out_rem > 0
        pct_out = outf[sub] * zfrac; out_streak = np.where(outf[sub], out_streak + 1, 0); load_z = np.where(outf, 0, load)[sub]
        # casualties / traffic
        cas = 25 * popr * np.maximum(0, dam[t] - dam_prev) + 4.0 * popr * np.clip(conc - 1, 0, 3); dam_prev = dam[t].copy()
        road_block = np.clip(0.7 * dam[t] + 0.7 * (conc > 1.5), 0, 0.9)
        speed_new = (1 - 1 / (1 + np.exp(-(depth - 12) / 4))) * (1 - 0.35 * pct_out) * (1 - 0.15 * rush[t]) * (1 - road_block)
        speed_new = np.clip(speed_new + rng.normal(0, 0.01, Z), 0.02, 1.0)
        # EMS
        mult = (0.65 + 0.35 * diur[t]) * (1 + 0.04 * depth + 0.05 * max(0, wbgt[t] - 28) + 0.2 * cas + 0.4 * np.minimum(conc, 5))
        calls = rng.poisson(base_calls * mult)
        eff = units * (1 - 0.5 * np.clip((occ.mean() - 0.8) / 0.15, 0, 1)) * (1 - 0.4 * dam[t].mean())
        rho = min(0.97, calls.sum() * 0.9 / eff); delay = 5 * rho ** 2 / (1 - rho)
        resp = 7 + 10 * (1 / np.maximum(speed, 0.2) - 1) + delay + rng.normal(0, 0.8, Z)
        speed = speed_new
        # epidemic / ED
        if 'epi' in comps:
            if t == t_seed: Ip[seeds] += 0.004
            force = R0 * gam * (0.85 * Ip + 0.15 * (W @ Ip)); newE = Sf * force
            Sf = Sf - newE; Ep = Ep + newE - sig * Ep; Ip = Ip + sig * Ep - gam * Ip; Ih[t] = Ip
        else: Ih[t] = I_bg
        I_lag = Ih[max(0, t - 12)]; I_adm = Ih[max(0, t - 48)]
        heat_exp = 0.9 * heat_exp + 0.1 * max(0, wbgt[t] - 26) ** 1.2
        ed_ratio = 1 + 120 * I_lag + 0.3 * heat_exp + 0.1 * cas + rng.normal(0, 0.02, Z)
        # hospitals
        ems_arr = 0.25 * (Cm @ calls); dive = occ > 0.9; moved = np.where(dive, 0.5 * ems_arr, 0.0); ems_arr = ems_arr - moved
        np.add.at(ems_arr, city['nn_h'], moved)
        arr = walkin * (0.9 + 0.2 * diur[t]) + ems_arr + B * 0.0006 * heat_exp + 70 * (Cm @ (popr * I_adm)) + Cm @ cas
        Npat = np.clip(Npat + arr - Npat / LOS, 0, 1.5 * B)
        gen_fail = out_streak[H] > city['gen_hours']; Beff = B * (1 - 0.4 * gen_fail); occ = np.clip(Npat / Beff, 0, 1.3)
        # severity for reports; communications
        Xt = np.zeros((Z, NV), np.float32)
        Xt[:, VI['rain_mm_h']] = rain[t]; Xt[:, VI['wbgt']] = wbgt[t]; Xt[:, VI['drain_load']] = np.clip(load_d, 0, 10)
        Xt[:, VI['water_depth_cm']] = depth; Xt[:, VI['speed_idx']] = speed; Xt[:, VI['load_ratio']] = load_z
        Xt[:, VI['pct_out']] = pct_out; Xt[:, VI['ems_resp_min']] = np.clip(resp, 2, 120); Xt[:, VI['bed_occ']] = occ[city['hos_of']]
        Xt[:, VI['ed_ratio']] = ed_ratio; Xt[:, VI['conc_idx']] = conc; Xt[:, VI['damage_idx']] = dam[t]
        Xt[:, VI['shaking_mmi']] = mmi_ts[t]; Xt[:, VI['temp_c']] = Tair[t]
        sev = stress_from_vars(Xt)[:, :9].max(1)
        reports = rng.poisson(0.3 + 8 * np.clip(sev, 0, 1.2)).astype(float)
        comload = 0.08 + 0.02 * calls + 0.025 * reports + 0.3 * pct_out
        comload = np.where(out_streak > 6, 1.2, comload)
        Xt[:, VI['report_rate']] = reports; Xt[:, VI['com_load']] = np.clip(comload, 0, 1.5)
        X[t] = Xt
    X = np.clip(X, VAR_MIN, VAR_MAX).astype(np.float32)
    G = G_EDGE_ARR.copy()
    return dict(X=X, fc=fc, fs_rain=fs_r, fs_temp=fs_t, G_edges=G, meta=meta)

# ----------------------------------------------------------------------------- 4. LABELS  (training / evaluation only)
def crit_conditions(X, lv):
    c = np.zeros(X.shape[:-1] + (7,), bool)
    c[..., 0] = lv[..., LI['TRF']] == 3; c[..., 1] = lv[..., LI['PWR']] == 3; c[..., 2] = lv[..., LI['HOS']] == 3
    c[..., 3] = (X[..., VI['wbgt']] >= 31.0) & (X[..., VI['ed_ratio']] >= 2.0)
    c[..., 4] = lv[..., LI['AIR']] == 3; c[..., 5] = lv[..., LI['SEI']] == 3; c[..., 6] = lv[..., LI['COM']] == 3
    return c
def sustained_onsets(cond, k=CRIT_K):
    T = cond.shape[0]; out = np.zeros_like(cond)
    for j in range(cond.shape[-1]):
        c = cond[..., j]; ok = c.copy()
        for s in range(1, int(k[j])):
            sh = np.zeros_like(c); sh[:T - s] = c[s:]; ok &= sh
        prev = np.zeros_like(c); prev[1:] = c[:-1]; out[..., j] = c & ~prev & ok
    return out
def cascade_series(onset, win=12, min_types=3, min_zones=2):
    T = onset.shape[0]; cs = np.cumsum(onset.astype(np.int32), axis=0)
    lag = np.zeros_like(cs); lag[win:] = cs[:T - win]; w = (cs - lag) > 0
    types = w.any(1).sum(-1); zones = w.any(2).sum(-1)
    state = (types >= min_types) & (zones >= min_zones); prev = np.zeros_like(state); prev[1:] = state[:-1]
    return state, state & ~prev
def label_episode(X):
    S = stress_from_vars(X); lv = levels_from_stress(S); cond = crit_conditions(X, lv); onset = sustained_onsets(cond)
    cst, con = cascade_series(onset)
    return dict(onset=onset.astype(np.uint8), crit_state=cond.astype(np.uint8), cas_state=cst.astype(np.uint8), cas_onset=con.astype(np.uint8))
def window_labels(onset_t, origins, horizons=HORIZONS):
    """onset_t (T,...) -> y (n_orig, len(h), ...) = any onset in (t, t+h]; only valid where t+max(h) < T"""
    cs = np.cumsum(onset_t.astype(np.int32), axis=0); T = cs.shape[0]
    y = np.stack([cs[np.minimum(origins + h, T - 1)] - cs[origins] > 0 for h in horizons], 1)
    return y

# ----------------------------------------------------------------------------- 5. observation model / faults
PROFILES = {'F0': {}, 'F1': dict(drop=(.2, .4), blackout=True), 'F2': dict(delay=True), 'F3': dict(bias=(.05, .10)),
            'F4': dict(adv=True), 'F5': dict(drop=(.1, .25), blackout=True, delay=True, bias=(.03, .07), adv=True)}
def observe(X, profile, seed, drop=None):
    """Return asof (T,Z,V) [NaN = never received], stale (T,Z,V) hours since event_time of the value known at as_of, info.
    event_time = t; ingest_time = t + delay.  Value known at as_of = the one with latest event_time among ingest_time<=as_of."""
    T, Z, V = X.shape; rng = np.random.default_rng(seed); P = PROFILES[profile]; info = dict(profile=profile)
    val = X + rng.normal(0, 1, X.shape).astype(np.float32) * NOISE_SIGMA
    D = np.tile(BASE_DELAY, (T, Z, 1)).astype(int); dropped = np.zeros(X.shape, bool)
    p_drop = drop if drop is not None else (rng.uniform(*P['drop']) if 'drop' in P else 0.0)
    if p_drop > 0: dropped |= rng.random(X.shape) < p_drop
    if P.get('blackout'):
        v0 = int(rng.integers(0, V)); t0 = int(rng.integers(48, 200)); dropped[t0:t0 + 6, :, v0] = True; info['blackout'] = (VARS[v0], t0, t0 + 6)
    if P.get('delay'):
        extra = np.ceil(rng.lognormal(np.log(2.0), 0.9, X.shape)).astype(int); heavy = rng.random(X.shape) < 0.08
        D = D + np.where(heavy, rng.integers(8, 25, X.shape), extra)
    if 'bias' in P:
        fr = rng.uniform(*P['bias']); bad = rng.random((Z, V)) < fr; stuck = bad & (rng.random((Z, V)) < 0.5); biased = bad & ~stuck
        rngv = (VAR_MAX - VAR_MIN) * 0.1
        val = val + (biased * rng.choice([-1, 1], (Z, V)) * rng.uniform(0.5, 1.5, (Z, V)) * rngv)[None]
        for z, v in zip(*np.where(stuck)):
            ts = int(rng.integers(40, 180)); val[ts:, z, v] = val[ts, z, v]
        info['n_bad_sensors'] = int(bad.sum())
    if P.get('adv'):
        zf = int(rng.integers(Z)); ta = int(rng.integers(60, 170)); w = int(rng.integers(12, 25))
        val[ta:ta + w, zf, VI['report_rate']] += rng.uniform(8, 14)                       # coordinated false citizen reports
        zs = rng.choice(Z, min(3, Z), replace=False); tb = int(rng.integers(60, 170))     # one coherently spoofed feed (traffic+depth)
        val[tb:tb + 24][:, zs, VI['speed_idx']] *= 0.35; val[tb:tb + 24][:, zs, VI['water_depth_cm']] += 20
        info.update(false_zone=zf, false_window=(ta, ta + w), spoof_zones=[int(z) for z in zs], spoof_window=(tb, tb + 24))
    val = np.clip(val, VAR_MIN, VAR_MAX)
    tt = np.arange(T)[:, None, None] * np.ones((1, Z, V), int)
    arrive = np.where(dropped, T, np.minimum(tt + D, T))
    zz = np.arange(Z)[None, :, None] * np.ones((T, 1, V), int); vv = np.arange(V)[None, None, :] * np.ones((T, Z, 1), int)
    M = np.full((T + 1, Z, V), -1, np.int32); np.maximum.at(M, (arrive, zz, vv), tt)
    K = np.maximum.accumulate(M[:T], axis=0)
    asof = np.take_along_axis(val, np.maximum(K, 0), axis=0).astype(np.float32); asof[K < 0] = np.nan
    stale = np.where(K >= 0, np.arange(T)[:, None, None] - K, 999).astype(np.float32)
    return asof, stale, info
def fill_defaults(asof):
    return np.where(np.isnan(asof), VAR_DEFAULT, asof).astype(np.float32)
def critical_active_obs(Xf):
    S = stress_from_vars(Xf); lv = levels_from_stress(S); return crit_conditions(Xf, lv).any(-1)

# ----------------------------------------------------------------------------- 6. events + Hawkes (numpy, analytic gradients)
def events_from_levels(lv, hold=3):
    """lv (T,Z,10) int levels -> N (T,Z,20) onsets of level>=2 ('deg') and ==3 ('fail') per layer.
    Debounced for noisy streams (causal): a condition is 'held' if true in any of the last `hold` hours; an onset is stamped
    at the 2nd consecutive held hour (confirmation), so single-hour sensor flicker never creates an event."""
    T, Z, _ = lv.shape; N = np.zeros((T, Z, NTYPE), np.uint8)
    for l in range(10):
        for k, c in enumerate((lv[..., l] >= 2, lv[..., l] == 3)):
            h = c.copy()
            for s_ in range(1, hold):
                h[s_:] |= c[:-s_]
            h1 = np.zeros_like(h); h1[1:] = h[:-1]; h2 = np.zeros_like(h); h2[2:] = h[:-2]
            N[..., 2 * l + k] = h & h1 & ~h2
    return N
def lag_conv(N, taus=TAUS):
    """C[t,z,b,j] = sum_{s<t} exp(-(t-s)/tau_b) N[s,z,j]   (strictly past; no instantaneous self-excitation)"""
    T, Z, J = N.shape; B = len(taus); C = np.zeros((T, Z, B, J), np.float32); d = np.exp(-1 / taus).astype(np.float32)
    acc = np.zeros((Z, B, J), np.float32); Nf = N.astype(np.float32)
    for t in range(1, T):
        acc = d[None, :, None] * (acc + Nf[t - 1][:, None, :]); C[t] = acc
    return C
def softplus(x): return np.logaddexp(0, x)
def sigmoid(x): return 1 / (1 + np.exp(-x))
def _rows(C, W):
    """stack list of episode tensors C (T,Z,B,J) -> rows (R,B,J) and spillover rows WC"""
    Cs = np.concatenate([c.reshape(-1, c.shape[2], c.shape[3]) for c in C]).astype(np.float32)
    WCs = np.concatenate([np.einsum('zy,tybj->tzbj', W, c).reshape(-1, c.shape[2], c.shape[3]) for c in C]).astype(np.float32)
    return Cs, WCs
def hawkes_lambda(A, mu, s, Cr, WCr):
    I, J, Bn = A.shape; Ct = (Cr + s[None, None, :] * WCr).reshape(len(Cr), -1)
    Af = A.transpose(0, 2, 1).reshape(I, Bn * J); return Ct @ Af.T + mu, Ct, Af
def hawkes_nll_grad(A, mu, s, Cr, WCr, Nr, w=None, need_grad=True):
    """mean Poisson NLL per row and gradients wrt A (I,J,B), mu (I), s (J). w: optional row weights (R,)"""
    lam, Ct, Af = hawkes_lambda(A, mu, s, Cr, WCr); lam = np.maximum(lam, 1e-8)
    w = np.ones(len(lam), np.float32) if w is None else w; ws = w.sum()
    nll = float((w[:, None] * (lam - Nr * np.log(lam))).sum() / ws)
    if not need_grad: return nll
    g = w[:, None] * (1 - Nr / lam) / ws; I, J, Bn = A.shape
    dAf = g.T @ Ct; dA = dAf.reshape(I, Bn, J).transpose(0, 2, 1); dmu = g.sum(0)
    gA = (g @ Af).reshape(len(g), Bn, J); ds = (gA * WCr).sum((0, 1))
    return nll, dA, dmu, ds
class Adam:
    def __init__(self, shapes, lr=0.02, b1=0.9, b2=0.999):
        self.m = [np.zeros(s) for s in shapes]; self.v = [np.zeros(s) for s in shapes]; self.t = 0; self.lr = lr; self.b1 = b1; self.b2 = b2
    def step(self, params, grads):
        self.t += 1
        for i, (p, g) in enumerate(zip(params, grads)):
            self.m[i] = self.b1 * self.m[i] + (1 - self.b1) * g; self.v[i] = self.b2 * self.v[i] + (1 - self.b2) * g * g
            mh = self.m[i] / (1 - self.b1 ** self.t); vh = self.v[i] / (1 - self.b2 ** self.t)
            p -= self.lr * mh / (np.sqrt(vh) + 1e-8)
def hawkes_reg(A, l1, lg):
    """L1 + per-edge group lasso (over lag basis).  returns value and dA"""
    nrm = np.sqrt((A ** 2).sum(2) + 1e-12)
    return l1 * A.sum() + lg * nrm.sum(), l1 + lg * A / nrm[:, :, None]
def branching_ratio(A, taus=TAUS, cross_only=True):
    """spectral radius of the integrated kernel. cross_only=True zeroes same-layer blocks (deg<->fail of one layer are trivially
    coupled), so rho measures propagation BETWEEN city systems -- the 'cascade reproduction number'."""
    Ai = (A * taus[None, None, :]).sum(2)
    if cross_only:
        Ai = Ai.copy()
        for i in range(NTYPE):
            for j in range(NTYPE):
                if i // 2 == j // 2: Ai[i, j] = 0
    return float(np.max(np.abs(np.linalg.eigvals(Ai)))), Ai
def layer_matrix(A, taus=TAUS):
    """integrated influence aggregated to layers; returns M[from_layer, to_layer]"""
    Ai = (A * taus[None, None, :]).sum(2)          # [target type, source type]
    M = np.zeros((10, 10))
    for i in range(NTYPE):
        for j in range(NTYPE): M[j // 2, i // 2] += Ai[i, j]
    return M
def layer_lag(A, taus=TAUS):
    """kernel-mean lag per layer pair (mass-weighted over exponential bases)"""
    mass = A * taus[None, None, :]; L = np.zeros((10, 10)); Mm = np.zeros((10, 10))
    for i in range(NTYPE):
        for j in range(NTYPE):
            m = mass[i, j]; Mm[j // 2, i // 2] += m.sum(); L[j // 2, i // 2] += (m * taus).sum()
    return L / np.maximum(Mm, 1e-12)
def hawkes_intensity_features(N, hk, W):
    """per-time, per-zone intensity of every type + 2nd-order 'expected downstream' intensity"""
    C = lag_conv(N, hk['taus']); WC = np.einsum('zy,tybj->tzbj', W, C); Ct = C + hk['s'][None, None, None, :] * WC
    T, Z = N.shape[:2]; I, J, Bn = hk['A'].shape
    lam = Ct.reshape(T * Z, -1) @ hk['A'].transpose(0, 2, 1).reshape(I, -1).T + hk['mu']; lam = lam.reshape(T, Z, I)
    Ai = (hk['A'] * hk['taus'][None, None, :]).sum(2); down = (lam - hk['mu']) @ Ai.T
    return lam.astype(np.float32), down.astype(np.float32)

# ----------------------------------------------------------------------------- 7. ETAS (MLE)
def etas_negll(theta_log, t, m, T_end, m0, window=None):
    mu, K, al, c, p = np.exp(theta_log[0]), np.exp(theta_log[1]), theta_log[2], np.exp(theta_log[3]), 1.0 + np.exp(theta_log[4])
    n = len(t); dt = t[:, None] - t[None, :]; mask = dt > 0
    if window is not None: mask &= dt < window
    prod = K * 10 ** (al * (m - m0))
    kern = np.where(mask, (np.where(mask, dt, 1.0) + c) ** (-p), 0.0)
    lam = mu + (kern * prod[None, :]).sum(1)
    Tw = T_end - t
    if window is not None: Tw = np.minimum(Tw, window)
    integ = mu * T_end + (prod * (c ** (1 - p) - (Tw + c) ** (1 - p)) / (p - 1)).sum()
    return -(np.log(lam).sum() - integ)
def etas_fit(t, m, T_end, m0, x0=None, window=None):
    from scipy.optimize import minimize
    x0 = np.array([np.log(len(t) / T_end * 0.3), np.log(0.01), 0.8, np.log(0.02), np.log(0.15)]) if x0 is None else x0
    bnds = [(-12, 4), (-9, 1), (0.05, 2.5), (-9, 0), (-4, 1.5)]
    r = minimize(etas_negll, x0, args=(t, m, T_end, m0, window), method='L-BFGS-B', bounds=bnds)
    th = r.x; return dict(mu=float(np.exp(th[0])), K=float(np.exp(th[1])), alpha=float(th[2]), c=float(np.exp(th[3])), p=float(1 + np.exp(th[4])), nll=float(r.fun), x=th, ok=bool(r.success))

# ----------------------------------------------------------------------------- 8. features for the risk head
def _shift(a, k):
    out = np.empty_like(a); out[k:] = a[:-k]; out[:k] = a[0:1]; return out
def build_features(asof, stale, fc, city, origins, hawkes=None, chronos=None, t_refresh=None):
    """Features at forecast origins (as-of discipline: every quantity at origin t only uses asof[:t+1])"""
    Xf = fill_defaults(asof); T, Z, _ = Xf.shape; S = stress_from_vars(Xf); lv = levels_from_stress(S); o = np.asarray(origins)
    cols, names, groups = [], [], []
    def add(arr, nm, grp):                                   # arr (n_orig, Z, k)
        arr = arr.reshape(len(o), Z, -1).astype(np.float32); cols.append(arr)
        names.extend(nm if isinstance(nm, list) else [nm]); groups.extend([grp] * arr.shape[2])
        assert len(names) == len(groups) == sum(c.shape[2] for c in cols), nm
    add(Xf[o], [f'v_{v}' for v in VARS], 'state'); add(S[o], [f's_{l}' for l in LAYERS], 'state')
    for k in (1, 3, 6): add(S[o] - S[np.maximum(o - k, 0)], [f'd{k}_{l}' for l in LAYERS], 'trend')
    ew = np.zeros_like(S); ew[0] = S[0]
    for t in range(1, T): ew[t] = 0.3 * S[t] + 0.7 * ew[t - 1]
    add(ew[o], [f'ewma_{l}' for l in LAYERS], 'trend')
    for w in (6, 24):
        idx = np.maximum(o[:, None] - np.arange(w)[None], 0); add(S[idx].max(1), [f'max{w}_{l}' for l in LAYERS], 'trend')
    nb = np.einsum('zy,tyl->tzl', city['W'], S); add(nb[o], [f'nb_{l}' for l in LAYERS], 'neighbor')
    add(np.einsum('zy,ty->tz', city['UP'], S[..., LI['HYD']])[o][..., None], 'up_hyd', 'neighbor')
    add(np.einsum('zy,ty->tz', city['SUB'], S[..., LI['PWR']])[o][..., None], 'feeder_pwr', 'neighbor')
    add(np.broadcast_to(S.mean(1, keepdims=True), S.shape)[o], [f'city_mean_{l}' for l in LAYERS], 'neighbor')
    add(np.broadcast_to(S.max(1, keepdims=True), S.shape)[o], [f'city_max_{l}' for l in LAYERS], 'neighbor')
    add(fc[o], ['fc_rain6', 'fc_rain24', 'fc_rain72', 'fc_texc6', 'fc_texc24', 'fc_texc72'], 'forecast_cov')
    # early-warning signals (Scheffer et al.): rolling variance, lag-1 autocorr over 24 h; 48 h trend slope
    idx = np.maximum(o[:, None] - np.arange(24)[None, ::-1], 0); win = S[idx]                  # (n,24,Z,10)
    var = win.var(1); a = win[:, 1:] - win[:, 1:].mean(1, keepdims=True); b = win[:, :-1] - win[:, :-1].mean(1, keepdims=True)
    ar1 = (a * b).sum(1) / np.sqrt((a ** 2).sum(1) * (b ** 2).sum(1) + 1e-9)
    add(var, [f'ewsvar_{l}' for l in LAYERS], 'ews'); add(ar1, [f'ewsar1_{l}' for l in LAYERS], 'ews')
    idx = np.maximum(o[:, None] - np.arange(48)[None, ::-1], 0); win = S[idx]; tc = np.arange(48) - 23.5
    add((win * tc[None, :, None, None]).sum(1) / (tc ** 2).sum(), [f'slope48_{l}' for l in LAYERS], 'ews')
    # data quality
    st = np.minimum(stale, 48); cov = (stale <= 3).mean(-1)
    add(np.stack([cov, st.mean(-1), st.max(-1), np.broadcast_to(cov.min(1, keepdims=True), cov.shape)], -1)[o], ['dq_cov', 'dq_stale_mean', 'dq_stale_max', 'dq_city_mincov'], 'dq')
    add(np.broadcast_to(city['static'][None], (len(o), Z, city['static'].shape[1])), [f'st_{n}' for n in STATIC_NAMES], 'static')
    # trigger ages (first detection, causal)
    def age(cond):
        cum = np.maximum.accumulate(cond); first = np.argmax(cond) if cond.any() else T + 999
        return np.where(o >= first, np.minimum(o - first, 99), -1).astype(np.float32)
    ag = np.stack([age((Xf[..., VI['damage_idx']] > 0.05).any(1) | (Xf[..., VI['shaking_mmi']] >= 5).any(1)),
                   age((Xf[..., VI['conc_idx']] > 0.5).any(1)), age((Xf[..., VI['rain_mm_h']] > 2).any(1)),
                   age((Xf[..., VI['wbgt']] > 28.5).any(1))], -1)
    add(np.broadcast_to(ag[:, None, :], (len(o), Z, 4)), ['age_quake', 'age_gas', 'age_rain', 'age_heat'], 'trigger')
    if hawkes is not None:
        Nev = events_from_levels(lv); lam, dn = hawkes_intensity_features(Nev, hawkes, city['W'])
        add(lam[o], [f'hk_lam_{e}' for e in EVENT_TYPES], 'hawkes'); add(dn[o], [f'hk_down_{e}' for e in EVENT_TYPES], 'hawkes')
        loop = ((lv[..., LI['EMS']] >= 2) & (lv[..., LI['HOS']] >= 2)).astype(np.float32)
        add(np.stack([loop, np.full_like(loop, hawkes['rho'])], -1)[o], ['hk_loop_flag', 'hk_rho'], 'hawkes')
    if chronos is not None:
        r = np.searchsorted(t_refresh, o, side='right') - 1; ok = r >= 0; r = np.maximum(r, 0)
        ch = chronos[r].astype(np.float32); ch[~ok] = np.nan; ch = ch.reshape(len(o), Z, -1)
        nm = [f'ch_{CH_TARGETS[i]}_h{h}_q{q}' for i in range(len(CH_TARGETS)) for h in CH_H for q in (10, 50, 90)]
        add(ch, nm, 'chronos'); add(np.broadcast_to((o - t_refresh[r])[:, None, None].astype(np.float32), (len(o), Z, 1)), ['chronos_age'], 'chronos')
    F = np.concatenate(cols, 2)
    return F, names, groups
CH_TARGETS = ['water_depth_cm', 'speed_idx', 'load_ratio', 'bed_occ', 'ed_ratio', 'wbgt']
CH_H = [6, 24, 72]

# ----------------------------------------------------------------------------- episode IO
def save_episode(path, sim, lab, meta):
    np.savez_compressed(path, X=sim['X'], fc=sim['fc'], fs_rain=sim['fs_rain'], fs_temp=sim['fs_temp'], G_edges=sim['G_edges'],
                        onset=lab['onset'], crit_state=lab['crit_state'], cas_state=lab['cas_state'], cas_onset=lab['cas_onset'],
                        meta=np.array(json.dumps(meta)))
def load_episode(path):
    d = dict(np.load(path, allow_pickle=False)); d['meta'] = json.loads(str(d['meta'])); return d

In [ ]:
import os, sys, glob, json, time, math, warnings, pickle
import numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
from tqdm.auto import tqdm
warnings.filterwarnings('ignore'); plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .25, 'figure.figsize': (9, 4)})
ON_KAGGLE = os.path.exists('/kaggle/working')
IN_ROOT = '/kaggle/input' if ON_KAGGLE else os.environ.get('W72_IN', './inputs')
OUT = '/kaggle/working' if ON_KAGGLE else os.environ.get('W72_OUT', './outputs')
FIG = f'{OUT}/figs'; os.makedirs(FIG, exist_ok=True)
def find(name, required=True):
    hits = sorted(glob.glob(f'{IN_ROOT}/**/{name}', recursive=True) + glob.glob(f'{OUT}/**/{name}', recursive=True))
    if not hits and required: raise FileNotFoundError(f'{name} not found under {IN_ROOT} - add the producing notebook/dataset via "Add Input"')
    return hits[0] if hits else None
def savefig(n): plt.tight_layout(); plt.savefig(f'{FIG}/{n}.png', bbox_inches='tight'); plt.show()
sys.path.insert(0, os.path.dirname(find('w72_common.py'))); import w72_common as W

In [ ]:
# ------------------------------------------------------------------ CONFIG
QUICK = False                      # True = tiny smoke run (~1 min) to validate the pipeline end-to-end
N = dict(train=500, val=100, test=100, heldout=100, drift=40)       # drift = N/2 paired (drifted, control) flood episodes
if QUICK: N = dict(train=24, val=9, test=9, heldout=9, drift=4)
T = 240; SEED0 = dict(train=100_000, val=200_000, test=300_000, heldout=400_000, drift=500_000)
DATA = f'{OUT}/data'; [os.makedirs(f'{DATA}/{s}', exist_ok=True) for s in N]
CITIES = {'anchor': W.make_city('anchor'), 'heldout': W.make_city('heldout')}
print({k: (v['Z'], v['nsub'], len(v['hosp'])) for k, v in CITIES.items()}, '(zones, substations, hospitals)')

In [ ]:
def plan(split, i):
    """deterministic episode plan -> (ep_id, city, hazard, seed, profile, drift_at)"""
    rng = np.random.default_rng(SEED0[split] + 7919 * i)
    if split == 'drift':
        half = N['drift'] // 2; pair = i % half; drifted = i < half
        return f'drift_{i:04d}', 'anchor', 'flood', SEED0['drift'] + pair, 'F0', (100 if drifted else None)
    hz = W.HAZARDS[rng.choice(len(W.HAZARDS), p=W.HAZARD_W)] if split == 'train' else W.HAZARDS[i % len(W.HAZARDS)]
    prof = W.PROFILE_NAMES[rng.choice(6, p=[.3, .14, .14, .14, .14, .14])] if split == 'train' else W.PROFILE_NAMES[(i + i // 9) % 6]
    return f'{split}_{i:04d}', ('heldout' if split == 'heldout' else 'anchor'), hz, SEED0[split] + i, prof, None

def make_episode(split, i):
    ep, city, hz, seed, prof, dr = plan(split, i); path = f'{DATA}/{split}/{ep}.npz'
    if os.path.exists(path): return path
    sim = W.simulate(CITIES[city], hz, seed, T=T, drift_at=dr); lab = W.label_episode(sim['X'])
    meta = dict(sim['meta'], ep_id=ep, split=split, profile=prof, hazard=hz, city=city, seed=int(seed),
                n_onsets=lab['onset'].sum((0, 1)).tolist(), cascade=int(lab['cas_onset'].sum() > 0))
    W.save_episode(path, sim, lab, meta); return path

t0 = time.time()
for split in N:
    for i in tqdm(range(N[split]), desc=f'simulate {split}'):
        make_episode(split, i)
print(f'done in {(time.time()-t0)/60:.1f} min')

In [ ]:
# manifest (rebuilt from the files, so it is always consistent with what is on disk)
rows = []
for split in N:
    for p in sorted(glob.glob(f'{DATA}/{split}/*.npz')):
        m = json.loads(str(np.load(p)['meta'])); on = m.pop('n_onsets')
        rows.append(dict(m, **{f'on_{n}': v for n, v in zip(W.CRIT_NAMES, on)}, path=os.path.relpath(p, OUT)))
man = pd.DataFrame(rows); man.to_csv(f'{OUT}/w72_manifest.csv', index=False)
print(man.groupby(['split', 'hazard']).size().unstack(fill_value=0))
print('\nprofiles per split:\n', man.groupby(['split', 'profile']).size().unstack(fill_value=0))

## Sanity checks — the simulator must be physically sane and the labels non-degenerate

In [ ]:
chk = []
for p in tqdm(man.path.sample(min(30, len(man)), random_state=0), desc='checks'):
    d = W.load_episode(f'{OUT}/{p}'); X = d['X']
    chk.append(dict(finite=bool(np.isfinite(X).all()), in_bounds=bool(((X >= W.VAR_MIN - 1e-3) & (X <= W.VAR_MAX + 1e-3)).all()),
                    fc_causal=bool(np.isfinite(d['fc']).all())))
print(pd.DataFrame(chk).mean())
crit_rate = man.groupby('hazard')[[f'on_{n}' for n in W.CRIT_NAMES]].agg(lambda s: (s > 0).mean())
crit_rate.columns = W.CRIT_NAMES
print('\nfraction of episodes with each CRITICAL event type, by hazard'); display(crit_rate.round(2))
print('\ncascade (>=3 critical types in 12 h, >=2 zones) rate by hazard'); print(man.groupby('hazard').cascade.mean().round(2).to_dict())
bg = crit_rate.loc['none'].max()
print('background (no-hazard) worst critical rate:', round(bg, 2), '(random equipment faults are intentional: they are what false alarms are measured against)')

## Plots

In [ ]:
# 1) dataset composition + label base rates
fig, ax = plt.subplots(1, 2, figsize=(14, 4.2))
ct = man.groupby(['split', 'hazard']).size().unstack(fill_value=0); ct.plot(kind='bar', stacked=True, ax=ax[0], width=.8, colormap='tab20'); ax[0].set_title('episodes per split / hazard'); ax[0].legend(fontsize=7, ncol=3)
im = ax[1].imshow(crit_rate.values, cmap='magma_r', vmin=0, vmax=1, aspect='auto'); ax[1].set_xticks(range(7)); ax[1].set_xticklabels(W.CRIT_NAMES, rotation=30)
ax[1].set_yticks(range(len(crit_rate))); ax[1].set_yticklabels(crit_rate.index); ax[1].grid(False); ax[1].set_title('P(critical event occurs) per hazard — label base rates')
for (i, j), v in np.ndenumerate(crit_rate.values): ax[1].text(j, i, f'{v:.2f}', ha='center', va='center', fontsize=7, color='w' if v > .5 else 'k')
plt.colorbar(im, ax=ax[1]); savefig('00_composition_and_base_rates')

In [ ]:
# 2) example episodes: zone-mean stress per layer, trigger and cascade onset
def show_episode(hz):
    r = man[(man.hazard == hz) & (man.cascade == 1)]
    if not len(r): r = man[man.hazard == hz]
    if not len(r): return
    r = r.iloc[0]
    d = W.load_episode(f'{OUT}/{r.path}'); S = W.stress_from_vars(d['X']); lv = W.levels_from_stress(S)
    fig, ax = plt.subplots(1, 2, figsize=(15, 4.3), gridspec_kw=dict(width_ratios=[1.3, 1]))
    for l, n in enumerate(W.LAYERS): ax[0].plot(S[..., l].max(1), label=n, lw=1.4)
    ax[0].axhline(.85, ls='--', c='k', lw=.8); ax[0].axhline(.6, ls=':', c='k', lw=.8)
    if r.trigger_t >= 0: ax[0].axvline(r.trigger_t, c='r', ls='--', label='trigger')
    for t in np.where(d['cas_onset'])[0]: ax[0].axvline(t, c='purple', lw=2, alpha=.6)
    ax[0].set_title(f'{r.ep_id} · {hz} · max-over-zones stress (purple = CASCADE onset)'); ax[0].set_xlabel('hour'); ax[0].legend(ncol=6, fontsize=7)
    ax[1].imshow(S.max(2).T, aspect='auto', cmap='inferno', vmin=0, vmax=1.2); ax[1].set_title('zone × time, worst-layer stress'); ax[1].set_xlabel('hour'); ax[1].set_ylabel('zone'); ax[1].grid(False)
    savefig(f'01_episode_{hz}')
for hz in ['flood', 'heat', 'compoundA', 'quake']: show_episode(hz)

In [ ]:
# 3) cascade ORDER: median hours from trigger to first onset of each critical type (this is the chain the model must discover)
rows = []
for p, hz, trig in zip(man.path, man.hazard, man.trigger_t):
    if hz in ('none',) or trig < 0: continue
    d = W.load_episode(f'{OUT}/{p}'); on = d['onset'].any(1)           # (T,7)
    for k, n in enumerate(W.CRIT_NAMES):
        if on[:, k].any(): rows.append(dict(hazard=hz, type=n, delay=int(np.argmax(on[:, k])) - trig))
od = pd.DataFrame(rows)
piv = od.groupby(['hazard', 'type']).delay.median().unstack().reindex(columns=W.CRIT_NAMES)
fig, ax = plt.subplots(figsize=(9, 4.5)); im = ax.imshow(piv.values, cmap='viridis', aspect='auto'); ax.grid(False)
ax.set_xticks(range(7)); ax.set_xticklabels(W.CRIT_NAMES, rotation=30); ax.set_yticks(range(len(piv))); ax.set_yticklabels(piv.index)
for (i, j), v in np.ndenumerate(piv.values):
    if np.isfinite(v): ax.text(j, i, f'{v:.0f}h', ha='center', va='center', color='w', fontsize=8)
plt.colorbar(im, label='median hours from trigger'); ax.set_title('stage timing: first critical onset relative to trigger'); savefig('02_cascade_ordering')

In [ ]:
# 4) hidden ground-truth coupling G_true (layer level) with lag windows -- used ONLY for scoring dependency discovery
G = W.g_true_matrix(); fig, ax = plt.subplots(figsize=(7, 6)); ax.imshow(G, cmap='Blues', vmin=0, vmax=1.4); ax.grid(False)
ax.set_xticks(range(10)); ax.set_xticklabels(W.LAYERS); ax.set_yticks(range(10)); ax.set_yticklabels(W.LAYERS); ax.set_xlabel('TO (affected)'); ax.set_ylabel('FROM (cause)')
for a, b, lo, hi, mech in W.GEDGES: ax.text(W.LI[b], W.LI[a], f'{lo}-{hi}h', ha='center', va='center', fontsize=6.5, color='w')
ax.set_title(f'G_true: {G.sum()} directed edges, lag window in hours'); savefig('03_G_true')
print(pd.Series([m for *_, m in W.GEDGES]).value_counts().to_dict(), ' (Rinaldi-style mechanism tags)')

In [ ]:
# 5) weather-forecast realism: issued forecasts must degrade with lead (never the realized future)
errs = np.full((len(man), 72), np.nan)
for n, p in enumerate(man.path.head(120)):
    d = W.load_episode(f'{OUT}/{p}'); rain = d['X'][..., W.VI['rain_mm_h']].mean(1)
    if rain.max() < 3: continue
    e = []
    for i, ti in enumerate(range(0, T, 6)):
        L = min(72, T - ti); e.append(np.r_[np.abs(d['fs_rain'][i, :L] - rain[ti:ti + L]), np.full(72 - L, np.nan)])
    errs[n] = np.nanmean(np.array(e), 0)
mae = np.nanmean(errs, 0); fig, ax = plt.subplots(figsize=(8, 3.8)); ax.plot(np.arange(72), mae, lw=2); ax.set_xlabel('forecast lead (h)'); ax.set_ylabel('mean |error| city-mean rain, mm/h')
ax.set_title('issued rain-forecast error grows with lead (covariates for Chronos/GBM are these, not the truth)'); savefig('04_forecast_error_vs_lead')

In [ ]:
# 6) fault profiles F0..F5 on one variable + data-health summary
d = W.load_episode(f'{OUT}/{man[man.hazard=="flood"].path.iloc[0]}'); X = d['X']; v = W.VI['water_depth_cm']; z = int(np.argmax(X[..., v].max(0)))
fig, axs = plt.subplots(2, 3, figsize=(15, 6), sharex=True, sharey=True); summ = []
for ax, pf in zip(axs.ravel(), W.PROFILE_NAMES):
    a, s, info = W.observe(X, pf, 123); ax.plot(X[:, z, v], c='k', lw=1, label='truth'); ax.plot(a[:, z, v], c='tab:red', lw=1, label='as-of view'); ax.set_title(f'{pf}  {info.get("blackout","")}')
    summ.append(dict(profile=pf, missing_pct=100 * np.isnan(a).mean(), mean_stale_h=np.minimum(s, 48).mean(), p90_stale_h=np.percentile(np.minimum(s, 48), 90)))
axs[0, 0].legend(); fig.suptitle(f'water depth, zone {z}: truth vs what the system can know at each as_of'); savefig('05_fault_profiles')
display(pd.DataFrame(summ).round(2))

In [ ]:
# 7) the two cities (generalization test = unseen city with different elevation profile, drains, feeders, hospitals, climate)
fig, axs = plt.subplots(1, 2, figsize=(13, 5))
for ax, (k, c) in zip(axs, CITIES.items()):
    sc = ax.scatter(*c['coords'].T, c=c['elev'], s=380, cmap='terrain', edgecolor='k'); plt.colorbar(sc, ax=ax, label='elevation (m)')
    for z in range(c['Z']):
        if c['down'][z] >= 0: ax.annotate('', xy=c['coords'][c['down'][z]], xytext=c['coords'][z], arrowprops=dict(arrowstyle='->', color='tab:blue', lw=1.2))
        ax.text(*c['coords'][z], f's{c["sub"][z]}', ha='center', va='center', fontsize=7)
    ax.scatter(*c['coords'][c['hosp']].T, marker='P', s=160, c='r', label='hospital'); ax.set_title(f'{k}: {c["Z"]} zones, {c["nsub"]} feeders, drain cap mean {c["drain_cap"].mean():.1f} mm/h, T0={c["Tbase"]}°C'); ax.legend(); ax.grid(False)
fig.suptitle('blue arrows = drainage flow · labels = feeder/substation id'); savefig('06_cities')

In [ ]:
# 8) drift pairs: same weather, drained capacity halves at t=100 in the 'drift' episodes
dm = man[man.split == 'drift']; half = len(dm) // 2
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for k, (lab, rows_) in enumerate([('drifted', dm.iloc[:half]), ('control', dm.iloc[half:])]):
    s = np.mean([W.stress_from_vars(W.load_episode(f'{OUT}/{p}')['X'])[..., W.LI['HYD']].max(1) for p in rows_.path], 0); ax[0].plot(s, label=f'{lab} HYD stress')
    s = np.mean([W.stress_from_vars(W.load_episode(f'{OUT}/{p}')['X'])[..., W.LI['TRF']].max(1) for p in rows_.path], 0); ax[1].plot(s, label=f'{lab} TRF stress')
for a in ax: a.axvline(100, c='r', ls='--'); a.legend(); a.set_xlabel('hour')
fig.suptitle('paired drift episodes (used by NB02 online Hawkes + NB05 online calibration)'); savefig('07_drift_pairs')

In [ ]:
json.dump(dict(N=N, T=T, files=len(man), created=time.strftime('%Y-%m-%d %H:%M')), open(f'{OUT}/w72_dataset_info.json', 'w'))
print('OUTPUT FILES -> w72_common.py, w72_manifest.csv, data/<split>/*.npz, figs/*.png')
print('NEXT: Save Version (Save & Run All) -> in NB01..NB05 use Add Input -> this notebook\'s output.')